# RunPod：测试 PoC A2 标准名称链路

此 Notebook 与训练流程解耦，用于加载已经训练好的 LoRA 并做交互式测试。默认读取本地 poc_a/outputs/runpod-full：优先加载 metrics.json 选中的最佳 checkpoint；也可以指定某个 epoch，或改为加载已经发布到 Hugging Face 的 adapter。

建议先复制到仓库外的 /workspace/runpod_model_testing.ipynb，再从上到下执行。A2 会先生成标准名称和实体标签；只有名称存在于冻结实体表且标签与数据库一致时才返回 AppID，否则安全返回空。Notebook 同时复核零训练 alias 和冻结 NO_MATCH 集。

In [ ]:
from __future__ import annotations

import getpass
import html
import json
import os
import re
import subprocess
import sys
from pathlib import Path
from typing import Any, Mapping, Sequence

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path('/workspace/qwen-steam-entity-linking')]
PROJECT_DIR = next((path.resolve() for path in candidates if (path / 'poc_a/configs/qwen3_8b_lora.yaml').is_file()), None)
if PROJECT_DIR is None:
    raise RuntimeError('未找到项目。请先把仓库克隆到 /workspace，再打开此 Notebook。')

POC_DIR = PROJECT_DIR / 'poc_a'
# 本地模式：留空 HF_ADAPTER_ID，读取训练输出。
LOCAL_RUN_DIR = POC_DIR / 'outputs/runpod-full'
CHECKPOINT_EPOCH: int | None = None  # None：优先使用 metrics.json 选中的 checkpoint，否则使用最新 epoch。

# Hugging Face 模式：填写 user-or-org/model 后会忽略 LOCAL_RUN_DIR 和 CHECKPOINT_EPOCH。
HF_ADAPTER_ID = ''

BATCH_SIZE = 16
MAX_INPUT_TOKENS = 256
MAX_NEW_TOKENS = 64
os.environ.setdefault('HF_HOME', '/workspace/.cache/huggingface')

def run(command: list[str]) -> None:
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

print('PROJECT_DIR =', PROJECT_DIR)
print('POC_DIR =', POC_DIR)
print('HF_HOME =', os.environ['HF_HOME'])

## 1. 检查 GPU 并安装推理依赖

与训练 Notebook 使用同一份 poc_a/requirements-cloud.txt，不会重新安装镜像自带的 PyTorch。

In [ ]:
run(['nvidia-smi'])
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(POC_DIR / 'requirements-cloud.txt')])
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'hf_transfer>=0.1.9,<1'])

## 2. 可选：注入 Hugging Face Token

公开模型通常无需 token。私有基础模型或私有 adapter 才需要；输入只进入当前 Kernel 的环境变量，不会写回 Notebook。

In [ ]:
hf_token = getpass.getpass('HF_TOKEN（不需要可直接回车）：').strip()
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN 已注入当前 Kernel；不会显示或写入文件。')
else:
    print('未设置 HF_TOKEN。')

## 3. 解析要测试的 adapter

本地模式的选择顺序是：显式 CHECKPOINT_EPOCH → metrics.json 中的最佳 checkpoint → 训练目录中 epoch 最大的 checkpoint。回退到最新 epoch 时会明确提示，因为它不一定是 alias 泛化最好的版本。

In [ ]:
def read_json(path: Path) -> dict[str, Any]:
    try:
        value = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError) as error:
        raise RuntimeError(f'无法读取 {path}: {error}') from error
    if not isinstance(value, dict):
        raise RuntimeError(f'{path} 必须是 JSON object')
    return value

def local_checkpoint_records(run_dir: Path) -> list[tuple[int, int, Path]]:
    records: list[tuple[int, int, Path]] = []
    for checkpoint in (run_dir / 'checkpoints').glob('checkpoint-*'):
        metadata_path = checkpoint / 'checkpoint_meta.json'
        if not checkpoint.is_dir() or not metadata_path.is_file():
            continue
        metadata = read_json(metadata_path)
        try:
            records.append((int(metadata['epoch']), int(metadata['global_step']), checkpoint.resolve()))
        except (KeyError, TypeError, ValueError) as error:
            raise RuntimeError(f'checkpoint metadata 无效：{metadata_path}') from error
    return sorted(records)

if HF_ADAPTER_ID.strip():
    SOURCE_KIND = 'huggingface'
    ADAPTER_SOURCE: str | Path = HF_ADAPTER_ID.strip()
    SELECTED_EPOCH: int | None = None
else:
    SOURCE_KIND = 'local'
    LOCAL_RUN_DIR = LOCAL_RUN_DIR.resolve()
    if not LOCAL_RUN_DIR.is_dir():
        raise RuntimeError(f'训练输出目录不存在：{LOCAL_RUN_DIR}。请修改 LOCAL_RUN_DIR，或填写 HF_ADAPTER_ID。')
    records = local_checkpoint_records(LOCAL_RUN_DIR)
    if not records:
        raise RuntimeError(f'{LOCAL_RUN_DIR} 中没有完整 checkpoint')

    chosen: tuple[int, int, Path] | None = None
    if CHECKPOINT_EPOCH is not None:
        chosen = next((record for record in records if record[0] == CHECKPOINT_EPOCH), None)
        if chosen is None:
            raise RuntimeError(f'没有 epoch={CHECKPOINT_EPOCH}；可用 epoch：{[record[0] for record in records]}')
    else:
        metrics_path = LOCAL_RUN_DIR / 'metrics.json'
        if metrics_path.is_file():
            selection = read_json(metrics_path).get('selection')
            if isinstance(selection, dict) and selection.get('checkpoint'):
                selected_path = Path(str(selection['checkpoint']))
                selected_path = selected_path if selected_path.is_absolute() else LOCAL_RUN_DIR / selected_path
                selected_path = selected_path.resolve()
                chosen = next((record for record in records if record[2] == selected_path), None)
                if chosen is None:
                    raise RuntimeError(f'metrics.json 选中的 checkpoint 不存在：{selected_path}')
        if chosen is None:
            chosen = records[-1]
            print('提示：没有可用的评测选择结果，暂时使用最新 epoch；它不一定是最佳泛化版本。')

    SELECTED_EPOCH, _, ADAPTER_SOURCE = chosen
    missing = [name for name in ('adapter_config.json', 'adapter_model.safetensors') if not (ADAPTER_SOURCE / name).is_file()]
    if missing:
        raise RuntimeError(f'{ADAPTER_SOURCE} 缺少 adapter 文件：{missing}')

print('SOURCE_KIND =', SOURCE_KIND)
print('ADAPTER_SOURCE =', ADAPTER_SOURCE)
if SELECTED_EPOCH is not None:
    print('SELECTED_EPOCH =', SELECTED_EPOCH)

## 4. 加载 tokenizer、基础模型和 LoRA

本地模式从 run_manifest.json 读取训练时固定的基础模型 revision；Hugging Face 模式从 adapter_config.json 读取。加载 tokenizer 后会验证全部 GAME token 与 `<NO_MATCH>` 都是原子 token。

In [ ]:
import torch
from peft import PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

scripts_path = str(POC_DIR / 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from build_training_data import PROMPT_STYLES as PROMPT_STYLE_ROWS
from predict import load_canonical_entities
from structured_output import NO_MATCH_TOKEN, normalize_entity_name, resolve_structured_response
from training_common import data_hashes

if not torch.cuda.is_available():
    raise RuntimeError('测试 Qwen3-8B 需要 CUDA GPU')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('当前 GPU 不支持 BF16')

HF_TOKEN = os.environ.get('HF_TOKEN') or None
TRUST_REMOTE_CODE = False
if SOURCE_KIND == 'local':
    manifest = read_json(LOCAL_RUN_DIR / 'run_manifest.json')
    model_info = manifest.get('model')
    if not isinstance(model_info, dict) or not model_info.get('id') or not model_info.get('revision'):
        raise RuntimeError('run_manifest.json 缺少基础模型 id 或固定 revision')
    MODEL_ID = str(model_info['id'])
    MODEL_REVISION = str(model_info['revision'])
    resolved_config_path = LOCAL_RUN_DIR / 'resolved_config.json'
    if not resolved_config_path.is_file():
        raise RuntimeError(f'训练输出缺少 resolved_config.json：{resolved_config_path}')
    resolved_config = read_json(resolved_config_path)
    if data_hashes(resolved_config) != manifest.get('data_sha256'):
        raise RuntimeError('当前项目数据与训练时的数据哈希不一致；请切换到训练所用 Git commit 后再测试。')
    TRUST_REMOTE_CODE = bool(resolved_config.get('model', {}).get('trust_remote_code', False))
else:
    peft_config = PeftConfig.from_pretrained(ADAPTER_SOURCE, token=HF_TOKEN)
    MODEL_ID = str(peft_config.base_model_name_or_path)
    MODEL_REVISION = getattr(peft_config, 'revision', None)

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_SOURCE,
    use_fast=True,
    token=HF_TOKEN,
)
if tokenizer.eos_token_id is None:
    raise RuntimeError('tokenizer 没有 EOS token')
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

ENTITY_TOKEN_PATTERN = re.compile(r'^<GAME_([0-9]+)>$')
entity_tokens = sorted(
    (token for token in tokenizer.get_added_vocab() if ENTITY_TOKEN_PATTERN.fullmatch(token)),
    key=lambda token: int(ENTITY_TOKEN_PATTERN.fullmatch(token).group(1)),
)
if not entity_tokens:
    raise RuntimeError('adapter tokenizer 中没有 <GAME_APPID> token；请勿使用基础模型 tokenizer')
entity_token_ids = [int(tokenizer.convert_tokens_to_ids(token)) for token in entity_tokens]
if len(entity_token_ids) != len(set(entity_token_ids)):
    raise RuntimeError('实体 token ID 不唯一')
for token, token_id in zip(entity_tokens, entity_token_ids):
    if tokenizer.encode(token, add_special_tokens=False) != [token_id]:
        raise RuntimeError(f'实体 token 不是原子 token：{token}')
no_match_token_id = int(tokenizer.convert_tokens_to_ids(NO_MATCH_TOKEN))
if tokenizer.encode(NO_MATCH_TOKEN, add_special_tokens=False) != [no_match_token_id]:
    raise RuntimeError('adapter tokenizer 缺少原子 <NO_MATCH> token')

model_kwargs: dict[str, Any] = {
    'trust_remote_code': TRUST_REMOTE_CODE,
    'token': HF_TOKEN,
    'torch_dtype': torch.bfloat16,
    'low_cpu_mem_usage': True,
}
if MODEL_REVISION:
    model_kwargs['revision'] = MODEL_REVISION
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
base_model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(base_model, ADAPTER_SOURCE, token=HF_TOKEN)
DEVICE = torch.device('cuda:0')
model.to(DEVICE)
model.eval()
model.config.use_cache = True
torch.backends.cuda.matmul.allow_tf32 = True

CANONICAL_BY_TOKEN = load_canonical_entities(PROJECT_DIR / 'common/data/steam_games.csv')

print('MODEL_ID =', MODEL_ID)
print('MODEL_REVISION =', MODEL_REVISION)
print('GPU =', torch.cuda.get_device_name(0))
print('entity_token_count =', len(entity_tokens))

## 5. 定义确定性批量预测

预测会确定性生成 `标准名称` 和 `Steam AppID` 两行，再用冻结实体表严格解析。未注册名称、名称与标签不一致、显式 NO_MATCH 或格式错误都不会返回 AppID。

In [ ]:
from IPython.display import HTML, display

# A2: generate canonical name + token, then strictly resolve both fields.
PROMPT_STYLES = dict(PROMPT_STYLE_ROWS)

def predict_prompt_rows(
    rows: Sequence[Mapping[str, Any]],
    *,
    batch_size: int = BATCH_SIZE,
    top_k: int = 5,
) -> list[dict[str, Any]]:
    del top_k
    if not rows:
        return []
    previous_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    results: list[dict[str, Any]] = []
    try:
        with torch.inference_mode():
            for offset in range(0, len(rows), batch_size):
                batch_rows = rows[offset:offset + batch_size]
                encoded = tokenizer(
                    [str(row['prompt']) for row in batch_rows],
                    add_special_tokens=False, padding=True, return_tensors='pt',
                )
                if int(encoded['attention_mask'].sum(dim=1).max()) > MAX_INPUT_TOKENS:
                    raise ValueError(f'输入超过 MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}')
                encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
                prompt_width = int(encoded['input_ids'].shape[1])
                generated = model.generate(
                    **encoded, do_sample=False, max_new_tokens=MAX_NEW_TOKENS,
                    eos_token_id=int(tokenizer.eos_token_id),
                    pad_token_id=int(tokenizer.pad_token_id),
                )
                for row, sequence in zip(batch_rows, generated.tolist()):
                    output_ids = []
                    for token_id in sequence[prompt_width:]:
                        if int(token_id) in {int(tokenizer.eos_token_id), int(tokenizer.pad_token_id)}:
                            break
                        output_ids.append(int(token_id))
                    raw_output = tokenizer.decode(
                        output_ids, skip_special_tokens=False,
                        clean_up_tokenization_spaces=False,
                    ).strip()
                    resolution = resolve_structured_response(raw_output, CANONICAL_BY_TOKEN)
                    expected = str(row.get('expected', ''))
                    expected_name = str(row.get('canonical_name', ''))
                    if expected:
                        generation_correct = (
                            resolution.resolved_token is None
                            if expected == NO_MATCH_TOKEN
                            else resolution.resolved_token == expected
                        )
                        canonical_correct = (
                            resolution.format_valid
                            and normalize_entity_name(resolution.generated_canonical_name)
                            == normalize_entity_name(expected_name)
                        )
                    else:
                        generation_correct = None
                        canonical_correct = None
                    appid = ''
                    if resolution.resolved_token:
                        appid = resolution.resolved_token.removeprefix('<GAME_').removesuffix('>')
                    results.append({
                        'input': str(row.get('input', row['prompt'])),
                        'prompt_style': str(row.get('prompt_style', 'custom')),
                        'expected': expected,
                        'expected_canonical_name': expected_name,
                        'generated_canonical_name': resolution.generated_canonical_name,
                        'generated_token': resolution.generated_token,
                        'resolved_token': resolution.resolved_token or '',
                        'appid': appid,
                        'status': resolution.status,
                        'format_valid': resolution.format_valid,
                        'canonical_name_correct': canonical_correct,
                        'generation_correct': generation_correct,
                        'raw_output': raw_output,
                        'type': str(row.get('type', 'custom')),
                    })
    finally:
        tokenizer.padding_side = previous_padding_side
    return results

def predict(
    inputs: Sequence[str], *, prompt_style: str = 'appid_question',
    batch_size: int = BATCH_SIZE, top_k: int = 5,
) -> list[dict[str, Any]]:
    if prompt_style not in PROMPT_STYLES:
        raise ValueError(f'未知 prompt_style：{prompt_style}')
    rows = [
        {'input': str(value), 'prompt_style': prompt_style,
         'prompt': PROMPT_STYLES[prompt_style].format(input_text=str(value))}
        for value in inputs
    ]
    return predict_prompt_rows(rows, batch_size=batch_size, top_k=top_k)

def show_results(
    rows: Sequence[Mapping[str, Any]],
    columns: Sequence[str] = (
        'input', 'expected_canonical_name', 'generated_canonical_name',
        'expected', 'generated_token', 'appid', 'status',
        'canonical_name_correct', 'generation_correct', 'raw_output',
    ),
) -> None:
    if not rows:
        print('没有结果。')
        return
    header = ''.join(f'<th>{html.escape(column)}</th>' for column in columns)
    body = []
    for row in rows:
        cells = ''.join(f'<td>{html.escape(str(row.get(column, "")))}</td>' for column in columns)
        body.append(f'<tr>{cells}</tr>')
    display(HTML('<table><style>th,td{border:1px solid #ccc;padding:6px}</style>'
                 f'<thead><tr>{header}</tr></thead><tbody>{"".join(body)}</tbody></table>'))

## 6. 单条或批量自由测试

直接修改 QUERIES。这里不提供 expected，适合观察别名、中文名和自然语言描述的泛化结果。

In [ ]:
QUERIES = [
    'CS2',
    '反恐精英2',
    '刀塔2',
    'Valve 的 MOBA 游戏 Dota',
    '绝地求生',
    'Palworld',
    '量子煎饼模拟器 3001',
    '我忘记名字的一款射击游戏',
]

quick_results = predict(QUERIES, prompt_style='appid_question', top_k=5)
show_results(quick_results)

## 7. 自定义带答案测试

expected 必须写成训练标签格式。Notebook 会从冻结实体表补出目标标准名称；generation_correct 表示严格名称/标签解析后是否得到正确实体。

In [ ]:
CUSTOM_CASES = [
    {'input': 'CS2', 'expected': '<GAME_730>'},
    {'input': '刀塔2', 'expected': '<GAME_570>'},
    {'input': 'PUBG', 'expected': '<GAME_578080>'},
    {'input': 'APEX', 'expected': '<GAME_1172470>'},
    {'input': 'Palworld', 'expected': '<GAME_1623730>'},
]

custom_rows = [
    {
        **case,
        'canonical_name': CANONICAL_BY_TOKEN[case['expected']],
        'prompt_style': 'appid_question',
        'prompt': PROMPT_STYLES['appid_question'].format(input_text=case['input']),
        'type': 'custom_labeled',
    }
    for case in CUSTOM_CASES
]
custom_results = predict_prompt_rows(custom_rows, top_k=5)
show_results(custom_results)
print('safe resolution accuracy =', sum(row['generation_correct'] for row in custom_results) / len(custom_results))

## 8. 快速复核冻结 alias 评测集

默认执行 184 个训练外 alias 输入 × 4 种 prompt，共 736 行。这里测试当前已加载的单个 checkpoint，不写入任何评测文件。

In [ ]:
RUN_FROZEN_ALIAS_EVAL = True

def print_breakdown(rows: Sequence[Mapping[str, Any]], key: str) -> None:
    for value in sorted({str(row[key]) for row in rows}):
        subset = [row for row in rows if str(row[key]) == value]
        accuracy = sum(bool(row['generation_correct']) for row in subset) / len(subset)
        print(f'{key}={value:<28} count={len(subset):>3}  safe_resolution={accuracy:.2%}')

if RUN_FROZEN_ALIAS_EVAL:
    alias_rows = []
    with (POC_DIR / 'data/eval_alias.jsonl').open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            value = json.loads(line)
            if not isinstance(value, dict):
                raise RuntimeError(f'eval_alias.jsonl:{line_number} 不是 JSON object')
            alias_rows.append(value)

    alias_results = predict_prompt_rows(alias_rows, batch_size=BATCH_SIZE, top_k=3)
    generation_accuracy = sum(row['generation_correct'] for row in alias_results) / len(alias_results)
    print(f'alias rows = {len(alias_results)}')
    print(f'alias safe resolution = {generation_accuracy:.2%}')
    print_breakdown(alias_results, 'type')
    print_breakdown(alias_results, 'prompt_style')

    alias_failures = [row for row in alias_results if not row['generation_correct']]
    print(f'failures = {len(alias_failures)}；下表最多展示前 50 条')
    show_results(alias_failures[:50])
else:
    print('已跳过冻结 alias 评测。')

RUN_FROZEN_UNKNOWN_EVAL = True
if RUN_FROZEN_UNKNOWN_EVAL:
    unknown_rows = [json.loads(line) for line in (POC_DIR / 'data/eval_unknown.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    unknown_results = predict_prompt_rows(unknown_rows, batch_size=BATCH_SIZE)
    rejection_accuracy = sum(row['generation_correct'] for row in unknown_results) / len(unknown_results)
    explicit_accuracy = sum(row['status'] == 'explicit_no_match' for row in unknown_results) / len(unknown_results)
    print(f'unknown safe rejection = {rejection_accuracy:.2%}')
    print(f'unknown explicit NO_MATCH = {explicit_accuracy:.2%}')
    show_results([row for row in unknown_results if not row['generation_correct']][:50])

## 9. 可选：运行正式全量评测

正式验收仍以 poc_a/scripts/evaluate.py 为准。它会重新加载并评测全部里程碑 checkpoint，覆盖 canonical、零训练 alias 与冻结 unknown 集，然后写入 metrics.json、checkpoint_comparison.csv 与 evaluation_failures.csv。此步骤耗时明显更长且只适用于本地训练输出，因此默认关闭。

In [ ]:
RUN_OFFICIAL_EVALUATION = False

if RUN_OFFICIAL_EVALUATION:
    if SOURCE_KIND != 'local':
        raise RuntimeError('正式全量评测需要本地 run directory；Hugging Face adapter 请使用上面的冻结 alias 复核。')
    run([
        sys.executable,
        str(POC_DIR / 'scripts/evaluate.py'),
        '--run-dir',
        str(LOCAL_RUN_DIR),
        '--all-milestones',
    ])
else:
    print('已跳过正式全量评测。')

if SOURCE_KIND == 'local' and (LOCAL_RUN_DIR / 'metrics.json').is_file():
    metrics = read_json(LOCAL_RUN_DIR / 'metrics.json')
    print('现有 acceptance_passed =', metrics.get('acceptance_passed'))
    print('现有 selection =', json.dumps(metrics.get('selection'), ensure_ascii=False, indent=2))